# 06 · Baseline models

So sánh DummyClassifier và Logistic Regression bằng cùng 5 fold `StratifiedGroupKFold`. Logistic Regression dùng pipeline với `MusicPreprocessor`; median, ngưỡng winsorize và scaler chỉ học từ phần fit của mỗi fold. Macro-F1 là metric chính theo workflow nguồn; accuracy là metric phụ.

In [1]:
from pathlib import Path
from datetime import datetime
import sys, csv, platform, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import clone

SEED = 42
RUN_START = datetime.now()
ROOT = Path(r"C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2")
DATA_DIR = ROOT / "DATA_CHALLENGE2"
LOG_DIR = ROOT / "logs"
NB_DIR = ROOT / "notebooks"
sys.path.insert(0, str(NB_DIR))
from preprocessing import MusicPreprocessor

TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test (2).csv"
TARGET = "Class"
ID = "Id"
GROUP = "group_id"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu raw: {TRAIN_PATH}")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = train[TARGET].astype(int)
X = train.drop(columns=[TARGET])
# Cùng nghệ sĩ + tên bài hát luôn thuộc một fold; không dùng nhãn để tạo group.
group_key = (train["Artist Name"].fillna("").astype(str).str.strip().str.lower()
             + "||" + train["Track Name"].fillna("").astype(str).str.strip().str.lower())
groups = pd.factorize(group_key)[0]
CV = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
SPLITS = list(CV.split(X, y, groups=groups))

def make_pipeline(model):
    # MusicPreprocessor chỉ fit trên từng phần train của fold.
    return Pipeline([("preprocess", MusicPreprocessor()), ("model", model)])

def append_run_log(stage, notebook, goal, steps, notes):
    end = datetime.now()
    elapsed = round((end - RUN_START).total_seconds(), 1)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    run_id = f"{stage}_{end:%Y%m%d_%H%M%S}"
    slug = "Tô-Doanh-Quang-Khu"
    md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}.md"
    suffix = 2
    while md_path.exists():
        md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}_{suffix}.md"
        suffix += 1
    record = {"run_id":run_id,"date":end.strftime("%Y-%m-%d"),"time":end.strftime("%H:%M:%S"),
        "runner":"Tô Doanh Quang Khu","stage":stage,"notebook":notebook,"goal":goal,
        "steps":steps,"duration_sec":elapsed,"python":sys.version.split()[0],
        "platform":platform.platform(),"notes":notes + f"; markdown_log={md_path.name}"}
    fields=["run_id","date","time","runner","stage","notebook","goal","steps","duration_sec","python","platform","notes"]
    exists=(LOG_DIR/"experiment_log.csv").exists() and (LOG_DIR/"experiment_log.csv").stat().st_size>0
    with (LOG_DIR/"experiment_log.csv").open("a",newline="",encoding="utf-8-sig") as f:
        writer=csv.DictWriter(f,fieldnames=fields)
        if not exists: writer.writeheader()
        writer.writerow(record)
    steps_md="\n".join(f"- {part.strip()}" for part in steps.split(";"))
    md=f"""# Nhật ký thí nghiệm – {stage}

| Mục | Giá trị |
|---|---|
| Người chạy | Tô Doanh Quang Khu |
| Ngày chạy | {end.strftime('%d/%m/%Y')} |
| Giờ bắt đầu – kết thúc | {RUN_START.strftime('%H:%M:%S')} – {end.strftime('%H:%M:%S')} ({elapsed} giây) |
| Notebook | `{notebook}` |
| Mục tiêu | {goal} |
| Môi trường | Python {sys.version.split()[0]} · {platform.platform()} |

## Các bước & kết quả

{steps_md}

## Nhận xét

{notes}
"""
    md_path.write_text(md,encoding="utf-8")
    print(f"Đã ghi CSV và Markdown: {md_path}")
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

models={
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression balanced": make_pipeline(LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED)),
}
rows=[]
for name,estimator in models.items():
    for fold,(tr,va) in enumerate(SPLITS,start=1):
        fitted=clone(estimator)
        fitted.fit(X.iloc[tr],y.iloc[tr])
        pred=fitted.predict(X.iloc[va])
        rows.append({"model":name,"fold":fold,"macro_f1":f1_score(y.iloc[va],pred,average="macro"),"accuracy":accuracy_score(y.iloc[va],pred)})
fold_results=pd.DataFrame(rows)
summary=fold_results.groupby("model").agg(macro_f1_mean=("macro_f1","mean"),macro_f1_std=("macro_f1","std"),accuracy_mean=("accuracy","mean")).sort_values("macro_f1_mean",ascending=False)
display(fold_results.pivot(index="fold",columns="model",values="macro_f1").round(4))
display(summary.round(4))

model,Dummy most_frequent,LogisticRegression balanced
fold,,
1,0.0392,0.5223
2,0.0392,0.5172
3,0.0392,0.5287
4,0.0392,0.5060
5,0.0392,0.5086


,macro_f1_mean,macro_f1_std,accuracy_mean
model,,,
LogisticRegression balanced,0.5166,0.0094,0.4503
Dummy most_frequent,0.0392,0.0000,0.2750


## Ghi nhật ký kết quả

In [2]:
summary_text="; ".join(f"{name}: macro-F1={r.macro_f1_mean:.6f} ± {r.macro_f1_std:.6f}, accuracy={r.accuracy_mean:.6f}" for name,r in summary.iterrows())
append_run_log("06_Baseline","06_Baseline.ipynb","Tạo mốc Dummy và Logistic Regression bằng stratified group CV",
    f"5-fold StratifiedGroupKFold; raw train={train.shape}; groups={pd.Series(groups).nunique()}; {summary_text}",
    "Mỗi fold fit preprocessing trong pipeline; test không dùng để chọn hoặc tính điểm")

Đã ghi CSV và Markdown: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\06_Baseline_20261112_191021_Tô-Doanh-Quang-Khu.md
